In [26]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

In [5]:
DATA_PATH = Path("data/processed/telco_features.csv")

df = pd.read_csv(DATA_PATH)

print("Dimensions du dataset :", df.shape)
print(df.head())

Dimensions du dataset : (7043, 38)
   customerID  gender  SeniorCitizen  Partner  Dependents  tenure  \
0  7590-VHVEG       0              0        1           0       1   
1  5575-GNVDE       1              0        0           0      34   
2  3668-QPYBK       1              0        0           0       2   
3  7795-CFOCW       1              0        0           0      45   
4  9237-HQITU       0              0        0           0       2   

   PhoneService  MultipleLines  OnlineSecurity  OnlineBackup  ...  \
0             0              0               0             1  ...   
1             1              0               1             0  ...   
2             1              0               1             1  ...   
3             0              0               1             0  ...   
4             1              0               0             0  ...   

   TechSupport_No internet service  TechSupport_Yes  StreamingTV_No  \
0                                0                0             

In [6]:
df = df.dropna(subset=["Churn"]).copy()


In [7]:
df["Churn"] = df["Churn"].astype(int)


In [8]:
X = df.drop(columns=["Churn", "customerID"], errors="ignore")
y = df["Churn"]

# Vérifier les dimensions
print("\nDimensions de X :", X.shape)
print("Dimensions de y :", y.shape)
print("\nRépartition de la target :")
print(y.value_counts(normalize=True))


Dimensions de X : (7043, 36)
Dimensions de y : (7043,)

Répartition de la target :
Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTrain :", X_train.shape)
print("Test  :", X_test.shape)


Train : (5634, 36)
Test  : (1409, 36)


In [10]:
numeric_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

# Pipeline pour les colonnes numériques
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [11]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [27]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

In [28]:
results = []
trained_models = {}

Path("models").mkdir(exist_ok=True)
Path("reports").mkdir(exist_ok=True)

for name, model in models.items():

    print(f"\n{'=' * 50}")
    print(f"Modèle : {name}")
    print("=" * 50)

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])


Modèle : Logistic Regression

Modèle : Random Forest

Modèle : XGBoost


In [31]:
results = []
trained_models = {}

Path("models").mkdir(exist_ok=True)
Path("reports").mkdir(exist_ok=True)

for name, model in models.items():

    print(f"\n{'=' * 50}")
    print(f"Modèle : {name}")
    print("=" * 50)

    try:
        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        # Entraîner le pipeline
        pipeline.fit(X_train, y_train)
        print("✅ Entraînement terminé")

        # Conserver le modèle entraîné
        trained_models[name] = pipeline

        # Évaluer le modèle
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1]

        results.append({
            "Model": name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(
                y_test, y_pred, zero_division=0
            ),
            "Recall": recall_score(
                y_test, y_pred, zero_division=0
            ),
            "F1": f1_score(y_test, y_pred, zero_division=0),
            "ROC_AUC": roc_auc_score(y_test, y_proba),
            "PR_AUC": average_precision_score(y_test, y_proba)
        })

        # Enregistrer chaque pipeline
        filename = name.lower().replace(" ", "_") + ".joblib"
        joblib.dump(pipeline, Path("models") / filename)
        print(f"💾 Modèle enregistré : {filename}")

    except Exception as e:
        print(f"❌ Erreur pour {name} : {type(e).__name__}: {e}")

results_df = pd.DataFrame(results)

if not results_df.empty:
    results_df = results_df.sort_values(
        "PR_AUC", ascending=False
    ).reset_index(drop=True)

    best_name = results_df.iloc[0]["Model"]
    joblib.dump(
        trained_models[best_name],
        Path("models") / "best_model.joblib"
    )

    results_df.to_csv(
        Path("reports") / "model_comparison.csv",
        index=False
    )

print("\nModèles entraînés :", list(trained_models.keys()))
display(results_df)


Modèle : Logistic Regression
✅ Entraînement terminé
💾 Modèle enregistré : logistic_regression.joblib

Modèle : Random Forest
✅ Entraînement terminé
💾 Modèle enregistré : random_forest.joblib

Modèle : XGBoost
✅ Entraînement terminé
💾 Modèle enregistré : xgboost.joblib

Modèles entraînés : ['Logistic Regression', 'Random Forest', 'XGBoost']


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,XGBoost,0.797019,0.641026,0.534759,0.583090,0.837548,0.644179
1,Logistic Regression,0.734564,0.500000,0.794118,0.613636,0.841099,0.632626
2,Random Forest,0.769340,0.557110,0.639037,0.595268,0.822271,0.611467


In [32]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]


In [34]:
metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_test, y_pred, zero_division=0
        ),
        "ROC_AUC": roc_auc_score(y_test, y_proba),
        "PR_AUC": average_precision_score(y_test, y_proba)
    }

results.append(metrics)
trained_models[name] = pipeline

print(pd.Series(metrics).to_string())
print("\nRapport de classification :")
print(classification_report(
        y_test, y_pred, zero_division=0
    ))

Model         XGBoost
Accuracy     0.797019
Precision    0.641026
Recall       0.534759
F1            0.58309
ROC_AUC      0.837548
PR_AUC       0.644179

Rapport de classification :
              precision    recall  f1-score   support

           0       0.84      0.89      0.87      1035
           1       0.64      0.53      0.58       374

    accuracy                           0.80      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409



In [35]:
ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        display_labels=["No churn", "Churn"]
    )
plt.title(f"Matrice de confusion - {name}")
plt.tight_layout()
plt.savefig(
        f"reports/confusion_matrix_{name.lower().replace(' ', '_')}.png"
    )
plt.close()


In [36]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="PR_AUC",
    ascending=False
)

print("\n===== COMPARAISON FINALE =====")
print(results_df.to_string(index=False))

results_df.to_csv(
    "reports/model_comparison.csv",
    index=False
)



===== COMPARAISON FINALE =====
              Model  Accuracy  Precision   Recall       F1  ROC_AUC   PR_AUC
            XGBoost  0.797019   0.641026 0.534759 0.583090 0.837548 0.644179
            XGBoost  0.797019   0.641026 0.534759 0.583090 0.837548 0.644179
            XGBoost  0.797019   0.641026 0.534759 0.583090 0.837548 0.644179
Logistic Regression  0.734564   0.500000 0.794118 0.613636 0.841099 0.632626
      Random Forest  0.769340   0.557110 0.639037 0.595268 0.822271 0.611467


In [37]:
print("Modèles présents dans trained_models :")
print(list(trained_models.keys()))

print("\nModèles présents dans results_df :")
print(results_df["Model"].tolist())

Modèles présents dans trained_models :
['Logistic Regression', 'Random Forest', 'XGBoost']

Modèles présents dans results_df :
['XGBoost', 'XGBoost', 'XGBoost', 'Logistic Regression', 'Random Forest']


In [38]:

from pathlib import Path
import re
import joblib

# Vérifier que l'entraînement a bien été exécuté
if "trained_models" not in globals() or not trained_models:
    raise RuntimeError(
        "Aucun modèle entraîné en mémoire. "
        "Exécute d'abord toutes les cellules d'entraînement."
    )

if "results_df" not in globals() or results_df.empty:
    raise RuntimeError(
        "Le tableau results_df est absent ou vide. "
        "Exécute les cellules de comparaison des modèles."
    )

# Créer le dossier de sauvegarde
models_dir = Path("models")
models_dir.mkdir(parents=True, exist_ok=True)

# Exporter tous les modèles entraînés
print("===== EXPORT DE TOUS LES MODÈLES =====")

for model_name, model in trained_models.items():
    safe_name = re.sub(
        r"[^a-zA-Z0-9_-]+",
        "_",
        str(model_name)
    ).strip("_").lower()

    model_path = models_dir / f"{safe_name}.joblib"

    joblib.dump(model, model_path)
    print(f"✓ {model_name} : {model_path}")

# Identifier le meilleur modèle selon PR-AUC
best_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_name]

# Sauvegarder une copie sous le nom attendu par l'API
best_path = models_dir / "best_model.joblib"
joblib.dump(best_model, best_path)

print("\n===== EXPORT TERMINÉ =====")
print(f"Meilleur modèle selon PR-AUC : {best_name}")
print(f"Modèle de référence : {best_path}")
print(f"Nombre de modèles exportés : {len(trained_models)}")


===== EXPORT DE TOUS LES MODÈLES =====
✓ Logistic Regression : models\logistic_regression.joblib
✓ Random Forest : models\random_forest.joblib
✓ XGBoost : models\xgboost.joblib

===== EXPORT TERMINÉ =====
Meilleur modèle selon PR-AUC : XGBoost
Modèle de référence : models\best_model.joblib
Nombre de modèles exportés : 3


In [39]:
best_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_name]

joblib.dump(best_model, "models/best_model.joblib")

print(f"\nMeilleur modèle selon PR-AUC : {best_name}")
print("Modèle sauvegardé dans models/best_model.joblib")



Meilleur modèle selon PR-AUC : XGBoost
Modèle sauvegardé dans models/best_model.joblib
